# Phenotype tables for the whole cohort (once per CDR)
One `{phenotype}.tsv` per phenotype in this run's list (person_id, phenotype, age, sex_at_birth), most recent value per person; only aggregate counts are printed.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
OUT_DIR = cfg.input("phenotype_tables")
plist = pd.read_csv(cfg.input("phenotype_list"), sep="\t", dtype=str)
todo = plist[plist["phenotype_name"].isin(cfg.lists["phenotypes"])]
bad = todo[~todo["source"].isin(["measurement", "measurement_unit_normalized"])]
assert bad.empty, f"not wired up: {list(bad.phenotype_name)}"
todo = [p for p in todo["phenotype_name"] if FORCE or not os.path.exists(f"{OUT_DIR}/{p}.tsv")]
print("to pull:", todo or "nothing")
setenv(OUT_DIR=OUT_DIR, PHENO_LIST=cfg.input("phenotype_list"), PHENOS=",".join(todo) or "-",
       REFERENCE_DATE=cfg.params.get("reference_date", "2024-07-01"))

In [ ]:
%%script Rscript -
for (p in c("data.table", "dplyr", "bigrquery", "allofus")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, repos = "https://cloud.r-project.org")
suppressMessages({ library(data.table); library(allofus) })
phenos <- strsplit(Sys.getenv("PHENOS"), ",")[[1]]
if (identical(phenos, "-")) quit(save = "no")
out_dir <- Sys.getenv("OUT_DIR"); dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
plist <- fread(Sys.getenv("PHENO_LIST"), sep = "\t", colClasses = "character")
con <- aou_connect()
for (ph in phenos) {
  row <- plist[phenotype_name == ph]
  q <- sprintf("
    WITH d AS (
      SELECT person_id, CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
        CASE WHEN sex_at_birth_concept_id = 45878463 THEN 'Female' WHEN sex_at_birth_concept_id = 45880669 THEN 'Male' ELSE 'Other' END AS sex_at_birth
      FROM {CDR}.person),
    m AS (
      SELECT person_id, value_as_number, unit_concept_id,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
      FROM {CDR}.measurement WHERE measurement_concept_id IN (%s) AND value_as_number IS NOT NULL)
    SELECT d.person_id, m.value_as_number AS phenotype, m.unit_concept_id, d.age, d.sex_at_birth
    FROM d INNER JOIN m ON d.person_id = m.person_id WHERE m.rn = 1", Sys.getenv("REFERENCE_DATE"), row$concept_id)
  x <- as.data.table(dplyr::collect(aou_sql(q)))
  x[, person_id := as.character(person_id)]
  if (row$source == "measurement_unit_normalized") x[unit_concept_id %in% 8636, phenotype := phenotype / 10]   # g/L -> g/dL
  x[, unit_concept_id := NULL]
  fwrite(x, file.path(out_dir, paste0(ph, ".tsv")), sep = "\t")
  cat(ph, ":", nrow(x), "people\n")
}